# Notebook Overview

This notebook automates the disaggregation of ward-level population data (or any other polygon-like dataset) into individual residential building footprints. By combining spatial boundaries, building footprint data, and official population projections, it assigns inhabitants to specific structures using two distinct methodologies:

**Standard Area-Based Distribution**: Allocates population proportionally based on the Gross Floor Area (GFA) of residential buildings, treating all residential space equally.

**Weighted Informal Settlement Distribution**: Applies a density weighting factor (currently set to 4) to buildings identified as "informal settlements", reflecting higher population densities in these areas.

The process ensures integer-level precision (avoiding fractional people) using a "largest remainder" method to distribute residuals accurately. The final output is a geospatial dataset (.parquet) containing building-level population estimates.

**NOTE. To run this notebook, 3 files are necessary with the following specifications:**
- **Buildings (.parquet recommended):** The footprints must have the following columns:
    - latitude: (Float) Used to recreate geometry if needed.
    - longitude: (Float) Used to recreate geometry if needed.
    - gfa_in_meters: (Float/Int) Gross Floor Area, used as the primary weight for distribution.
    - prediction: (String) Must contain values like "Residential", "Non-Residential", or "Industrial". The script specifically filters for "Residential".
    - settlement_classification: Must contain values like "Informal" or "Formal" to apply the density weighting.
- **Wards (.geojson or .shp):** Format: Polygon geometry delimiting the wards.
- **Population counts (.xlsx):** Information with the projected population count.

**The script was build around the provided information from the local authorities, thus, the naming of the columns and data structure might be adapted for new geographies/datasets.**


# Script

### Importing libraries

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import math

### Importing building footprints and wards

In [2]:
ward_boundaries_gdf = gpd.read_file(r"Ward_Census.shp")
buildings_gdf = gpd.read_parquet(r"Buildings.parquet")

**Checking columns in buildings dataset:** It must be present, longitude, latitude, gfa, prediction (Residential, Non-Residential or Industrial) and settlement classification (informal/formal).

In [3]:
buildings_gdf.head(3)

,longitude,latitude,gfa_in_meters,prediction,settlement_clasification,geometry
0,72.118268,23.844193,2.56881,Residential,informal,"POLYGON ((72.11827 23.84418, 72.11827 23.8442,..."
1,72.113039,23.854728,2.60115,Residential,None,"POLYGON ((72.11305 23.85472, 72.11305 23.85473..."
2,72.118336,23.851261,5.59328,Residential,informal,"POLYGON ((72.11833 23.85127, 72.11833 23.85125..."


### Column and datasets renaming

**Adapting cells to Format**: The main script was developed based on a slighly different data structure, thus, the need to adapt it to that format.

In [4]:
ward_boundaries_gdf = ward_boundaries_gdf[ward_boundaries_gdf['Level']== "WARD"]
ward_boundaries_gdf['Ward_Number_str'] = (
    ward_boundaries_gdf['Name']
    .str.extract(r'WARD NO\.-(\d+)', expand=False)
)
ward_boundaries_gdf['Name'] = ward_boundaries_gdf['Ward_Number_str'].astype(int)
ward_boundaries_gdf['Name'].astype(str)
ward_boundaries_gdf ['Ward_Number_str'].astype(str)
gdf_boundaries = ward_boundaries_gdf.copy()

In [5]:
ward_boundaries_gdf.head(3)

,id,WardNo,Level,Name,TOT_P,geometry,Ward_Number_str
0,6,6,WARD,6,13543,"MULTIPOLYGON (((72.10577 23.8363, 72.10572 23....",0006
1,5,5,WARD,5,4769,"POLYGON ((72.10624 23.84454, 72.10608 23.84457...",0005
2,11,11,WARD,11,20857,"POLYGON ((72.13693 23.8608, 72.13693 23.8608, ...",0011


Filtering out non-residential data and reprojecting to a known CRS.

In [6]:
buildings = buildings_gdf[buildings_gdf["prediction"] == "Residential"].copy()
other_buildings= buildings_gdf[buildings_gdf["prediction"] != "Residential"].copy()
buildings["geometry2"] = buildings_gdf["geometry"]
buildings = gpd.GeoDataFrame(
    buildings,
    geometry=gpd.points_from_xy(buildings["longitude"], buildings["latitude"]),
    crs="EPSG:4326"
)

Performing geospatial operation to join ward information to Buildings

In [7]:
joined = gpd.sjoin(buildings, gdf_boundaries, how="left", predicate="within")
joined_wo_nan = joined[joined['Name'].notna()]
buildings = joined_wo_nan

Importing excel file with population counts. Same procedure to adapt data.

In [8]:
#file from MHT
populationv2 = pd.read_excel(r"projections_final.xlsx")
populationv2 = populationv2[populationv2['Level']== "WARD"]
populationv2['Ward_Number_str'] = (
    populationv2['Name']
    .str.extract(r'WARD NO\.-(\d+)', expand=False)
)
populationv2['Name'] = populationv2['Ward_Number_str'].astype(int)
populationv2['Name'].astype(str)
populationv2 ['Ward_Number_str'].astype(str)
populationv2 ['WardNo'] = populationv2['Name']
population = populationv2.copy()

In [9]:
population.head(3)

,id,WardNo,Level,Name,TOT_P,Pop_2025,Unnamed: 6,Ward_Number_str
0,6.0,6,WARD,6,13543.0,19640.0586,NaN,0006
1,5.0,5,WARD,5,4769.0,6916.0038,NaN,0005
2,11.0,11,WARD,11,20857.0,30246.8214,NaN,0011


### Population breakdown

In [10]:
#total ground floor area
total_gfa = buildings['gfa_in_meters'].sum()
total_gfa

4190887.92911

In [11]:
#average gfa per inhabitant 
total_population = population["Pop_2025"].sum()
average_area_per_inhabitant = total_gfa / total_population
average_area_per_inhabitant

21.60859698292588

It distributes a total population among all the buildings in a district (a "ward"), with one critical rule: you can't have fractions of a person. It must assign whole numbers (integers) that add up perfectly to the ward's total population, and it needs a fair way to decide who gets the "leftover" people from the decimal remainders.

In [12]:
#without decimal numbers
buildings = buildings.copy()
buildings["inhabitants_with_integer_estimate"] = np.nan

for ward in range(1, 15):
    try:
        mask = buildings["Name"].astype(int) == ward
    except:
        mask = buildings["Name"].astype(str) == str(ward)
    
    df_ward = buildings.loc[mask].copy()
    
    try:
        pop_row = population[population["WardNo"].astype(int) == ward]
    except:
        pop_row = population[population["WardNo"].astype(str) == str(ward)]
    
    if df_ward.empty or pop_row.empty:
        continue
    
    total_pop = int(round(pop_row["Pop_2025"].values[0]))
    total_gfa = df_ward["gfa_in_meters"].sum()
    if total_gfa == 0:
        continue

    df_ward["ROcc"] = (df_ward["gfa_in_meters"] / total_gfa) * total_pop
    df_ward["IOcc"] = np.floor(df_ward["ROcc"]).astype(int)
    df_ward["FOcc"] = df_ward["ROcc"] - df_ward["IOcc"]

    deficit = int(round(total_pop - df_ward["IOcc"].sum()))
    if deficit <= 0:
        buildings.loc[mask, "inhabitants_with_integer_estimate"] = df_ward["IOcc"].values
        continue

    df_ward["DeserveFactor"] = np.where(df_ward["IOcc"] > 0, df_ward["FOcc"] / df_ward["IOcc"], 1.0)

    top_idx = df_ward["DeserveFactor"].nlargest(deficit).index
    df_ward.loc[top_idx, "IOcc"] += 1

    buildings.loc[mask, "inhabitants_with_integer_estimate"] = df_ward["IOcc"].values


It does the same, but addst an informal_multiplier in buildings that are classified as "informal".

In [13]:
#without decimal numbers + informal settlements constant
buildings = buildings.copy()
buildings["inhabitants_with_integer_informal"] = np.nan

informal_multiplier = 4

for ward in range(1, 15):
    # mask = buildings["Name"].astype(str) == str(ward)
    # df_ward = buildings.loc[mask].copy()
    # pop_row = population[population["WardNo"] == ward]
    try:
        mask = buildings["Name"].astype(int) == ward
    except:
        mask = buildings["Name"].astype(str) == str(ward)
    
    df_ward = buildings.loc[mask].copy()
    #print(len(df_ward))
    
    try:
        pop_row = population[population["WardNo"].astype(int) == ward]
    except:
        pop_row = population[population["WardNo"].astype(str) == str(ward)]
    
           
    if df_ward.empty or pop_row.empty:
        continue
    
    total_pop = int(round(pop_row["Pop_2025"].values[0]))
    total_gfa = df_ward["gfa_in_meters"].sum()
    #print(total_gfa)
    if total_gfa == 0:
        continue
    
    df_ward["weight"] = np.where(df_ward["settlement_clasification"].str.lower() == "informal", informal_multiplier, 1)
    df_ward["weighted_gfa"] = df_ward["gfa_in_meters"] * df_ward["weight"]
    total_weighted_gfa = df_ward["weighted_gfa"].sum()
    

    df_ward["ROcc"] = (df_ward["weighted_gfa"] / total_weighted_gfa) * total_pop
    

    df_ward["IOcc"] = np.floor(df_ward["ROcc"]).astype(int)
    df_ward["FOcc"] = df_ward["ROcc"] - df_ward["IOcc"]
    
    deficit = int(round(total_pop - df_ward["IOcc"].sum()))
    if deficit <= 0:
        buildings.loc[mask, "inhabitants_with_integer_informal"] = df_ward["IOcc"].values
        continue

    df_ward["DeserveFactor"] = np.where(df_ward["IOcc"] > 0, df_ward["FOcc"] / df_ward["IOcc"], 1)
    
    top_idx = df_ward["DeserveFactor"].nlargest(deficit).index
    df_ward.loc[top_idx, "IOcc"] += 1
    

    buildings.loc[mask, "inhabitants_with_integer_informal"] = df_ward["IOcc"].values


In [14]:
#adding back geometry + nonres buildings
buildings['geometry'] = buildings['geometry2'] 
merged = pd.concat([buildings, other_buildings], ignore_index=True)
merged_gdf = gpd.GeoDataFrame(merged, geometry='geometry')
merged_gdf = merged_gdf.to_crs(epsg=4326)
merged_gdf.to_parquet("Patan_population_breakdown_in_wards.parquet")

**Checking population differences:**

In [15]:
print(population['Pop_2025'].sum())
print(merged_gdf["inhabitants_with_integer_estimate"].sum())
print(merged_gdf["inhabitants_with_integer_informal"].sum())

193945.39739999996
193944.0
193944.0


In [16]:
buildings_out_of_wards = joined[~joined['Name'].notna()]

### Saving output

In [17]:
output = "Buildings_with_population_breakdown.parquet"

In [18]:
# Ensure CRS is consistent
for gdf in [buildings, other_buildings, buildings_out_of_wards]:
    gdf.set_crs("EPSG:4326", inplace=True, allow_override=True)

# Merge all subsets 
merged = pd.concat([buildings, other_buildings, buildings_out_of_wards], ignore_index=True)
merged = merged.drop_duplicates(subset='geometry')

columns_to_keep = ["longitude", 
                   "latitude", 
                   "gfa_in_meters", 
                   "prediction", 
                   "settlement_clasification", 
                   "geometry2", 
                   "inhabitants_with_integer_estimate",
                   "inhabitants_with_integer_informal"
                  ]

# Filter the dataframe first, then convert to GeoDataFrame
merged = gpd.GeoDataFrame(
    merged[columns_to_keep], 
    geometry="geometry2", 
    crs="EPSG:4326"
)

merged = merged.rename(columns={"geometry2": "geometry"})

# Verify count and save
print("Total merged buildings:", len(merged))

merged.to_parquet(output, index=False, engine="pyarrow")

print(f"Saved all buildings to: {output}")

C:\Users\renec\AppData\Roaming\Python\Python312\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


Total merged buildings: 182313
Saved all buildings to: Buildings_with_population_breakdown.parquet


In [19]:
merged.head(3)

,longitude,latitude,gfa_in_meters,prediction,settlement_clasification,geometry,inhabitants_with_integer_estimate,inhabitants_with_integer_informal
0,72.118268,23.844193,2.56881,Residential,informal,"POLYGON ((72.11827 23.84418, 72.11827 23.8442,...",1.0,1.0
1,72.113039,23.854728,2.60115,Residential,None,"POLYGON ((72.11305 23.85472, 72.11305 23.85473...",1.0,1.0
2,72.118336,23.851261,5.59328,Residential,informal,"POLYGON ((72.11833 23.85127, 72.11833 23.85125...",1.0,1.0
